# 21 — Normalization and Dropout

Normalization controls activation scale, while dropout injects random masking during training. Both can affect optimization and generalization, but they operate differently and have different training/inference behavior.

This notebook keeps the practice focused: batch normalization is implemented for two-dimensional activations, other normalization families are compared conceptually, and inverted dropout is implemented with a small expectation experiment.

## Learning objectives

- distinguish batch, layer, instance, and group normalization;
- implement batch-normalization forward and backward passes;
- explain learned scale $\gamma$ and shift $\beta$;
- distinguish training statistics from running inference statistics;
- implement inverted dropout;
- explain why dropout is stochastic only during training.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from cs231n_practice.gradient_check import (
    eval_numerical_gradient_array,
    relative_error,
)

SEED = 42
np.set_printoptions(precision=4, suppress=True)

## 1. What does normalization normalize?

A normalization layer chooses a set of values, calculates their mean and variance, standardizes them, and usually applies a learned scale and shift. The main distinction between normalization families is **which axes belong to that set**.

For convolutional activations $X:(N,C,H,W)$:

- **Batch normalization** calculates one mean and variance per channel using the batch and spatial axes $(N,H,W)$. It depends on other examples during training and uses running statistics during inference.
- **Layer normalization** normalizes each example using all of its feature axes $(C,H,W)$. Examples do not affect one another, and training and inference use the same current-example statistics.
- **Instance normalization** calculates statistics separately for every example and channel using only $(H,W)$. It is common when per-image appearance or style variation should be removed.
- **Group normalization** splits channels into groups and normalizes $(C_{group},H,W)$ separately within each example. It does not depend on batch size and lies between layer and instance normalization.

These methods do not merely differ by name: changing the axes changes which activations influence one another.

### Exercise 1 — Identify the retained statistics shapes

For `x_nchw` with shape `(N, C, H, W)`, calculate means with `keepdims=True`. Complete the axes for batch, layer, and instance normalization. Group normalization is shown after reshaping channels into two groups.

In [2]:
x_nchw = np.arange(2 * 4 * 3 * 3, dtype=np.float64).reshape(2, 4, 3, 3)

# TODO: choose the axes described above.
batch_mean = x_nchw.mean(axis=(0,2,3), keepdims=True)
layer_mean = x_nchw.mean(axis=(1,2,3), keepdims=True)
instance_mean = x_nchw.mean(axis=(2,3), keepdims=True)

grouped = x_nchw.reshape(2, 2, 2, 3, 3)  # (N, groups, C_per_group, H, W)
group_mean = grouped.mean(axis=(2, 3, 4), keepdims=True)

assert batch_mean.shape == (1, 4, 1, 1)
assert layer_mean.shape == (2, 1, 1, 1)
assert instance_mean.shape == (2, 4, 1, 1)
assert group_mean.shape == (2, 2, 1, 1, 1)
{
    "batch": batch_mean.shape,
    "layer": layer_mean.shape,
    "instance": instance_mean.shape,
    "group": group_mean.shape,
}

{'batch': (1, 4, 1, 1),
 'layer': (2, 1, 1, 1),
 'instance': (2, 4, 1, 1),
 'group': (2, 2, 1, 1, 1)}

## 2. Batch normalization

For affine activations $X:(N,D)$, batch normalization computes one statistic per feature using the $N$ examples:

$$
\mu_d=\frac{1}{N}\sum_n X_{n,d},
$$

$$
\sigma_d^2=\frac{1}{N}\sum_n(X_{n,d}-\mu_d)^2,
$$

$$
\hat{X}_{n,d}=\frac{X_{n,d}-\mu_d}{\sqrt{\sigma_d^2+\epsilon}},
$$

$$
Y_{n,d}=\gamma_d\hat{X}_{n,d}+\beta_d.
$$

$\epsilon$ prevents division by zero. Learned $\gamma$ and $\beta$ let the layer restore or change the standardized scale and offset when that helps the model.

### Training and inference

During training, the layer uses the current minibatch mean and variance and updates exponential running estimates:

$$
\text{running mean}\leftarrow m(\text{running mean})+(1-m)\mu,
$$

with the same pattern for variance. Here $m$ is momentum for the running statistic, not optimizer momentum.

During inference, predictions should not depend on which other examples happen to share a batch. The layer therefore uses the stored running mean and variance. $\gamma$ and $\beta$ are used in both modes.

### Exercise 2 — Implement batch normalization

Complete the training and inference branches. The mutable `state` dictionary stores running statistics and configuration. Return a cache only during training because inference does not need a backward pass.

In [6]:
def batchnorm_forward(x, gamma, beta, state):
    """Return batch-normalized output and a training cache."""
    mode = state.get("mode", "train")
    epsilon = state.get("epsilon", 1e-5)
    momentum = state.get("momentum", 0.9)
    running_mean = state.get("running_mean", np.zeros(x.shape[1]))
    running_variance = state.get("running_variance", np.zeros(x.shape[1]))

    if mode == "train":
        # compute batch statistics, normalize, scale, and shift.
        mean = x.mean(axis=0, keepdims=True)
        variance = x.var(axis=0, keepdims=True)
        inverse_std = 1.0 / np.sqrt(variance + epsilon)
        normalized = (x - mean) * inverse_std
        output = gamma * normalized + beta

        state["running_mean"] = momentum * running_mean + (1 - momentum) * mean
        state["running_variance"] = momentum * running_variance + (1 - momentum) * variance
        cache = (normalized, gamma, x - mean, inverse_std)
    elif mode == "test":
        # normalize using stored statistics; do not update state.
        normalized = (
            x - running_mean
        ) / np.sqrt(running_variance + epsilon)
        output = gamma * normalized + beta
        cache = None
    else:
        raise ValueError("mode must be 'train' or 'test'")

    return output, cache


batch_x = np.array([[1.0, 10.0], [3.0, 14.0], [5.0, 18.0]])
batch_gamma = np.ones(2)
batch_beta = np.zeros(2)
batch_state = {"mode": "train", "momentum": 0.9}
batch_output, batch_cache = batchnorm_forward(
    batch_x, batch_gamma, batch_beta, batch_state
)
assert np.allclose(batch_output.mean(axis=0), 0.0, atol=1e-7)
assert np.allclose(batch_output.var(axis=0), 1.0, atol=1e-5)
assert batch_cache is not None
batch_output

array([[-1.2247, -1.2247],
       [ 0.    ,  0.    ],
       [ 1.2247,  1.2247]])

## 3. Batch-normalization backward pass

Let `dout` be $\partial L/\partial Y$. Scale and shift give

$$
d\beta=\sum_n dY_n,
$$

$$
d\gamma=\sum_n dY_n\hat{X}_n.
$$

A compact expression for the input gradient is

$$
dX=\frac{\gamma(\sigma^2+\epsilon)^{-1/2}}{N}
\left[N dY-\sum_n dY-\hat{X}\sum_n(dY\hat{X})\right].
$$

The sums are per feature and keep the feature dimension. This compact formula is algebraically equivalent to backpropagating through mean, variance, normalization, scale, and shift as separate graph operations.

### Exercise 3 — Implement and check the backward pass

Complete the three gradients using the compact formula, then compare them with numerical gradients. Each numerical forward call uses a fresh state so running-statistic mutation cannot affect the result.

In [8]:
def batchnorm_backward(dout, cache):
    """Backpropagate through training-mode batch normalization."""
    normalized, gamma, _, inverse_std = cache
    num_examples = dout.shape[0]

    # Sum over the batch axis and keep one value per feature.
    dbeta = dout.sum(axis=0)
    dgamma = (dout * normalized).sum(axis=0)

    sum_dout = dout.sum(axis=0)
    sum_dout_normalized = (dout * normalized).sum(axis=0)
    dx = (gamma * inverse_std / num_examples) * (
        num_examples * dout
        - sum_dout
        - normalized * sum_dout_normalized
    )
    return dx, dgamma, dbeta


generator = np.random.default_rng(SEED)
check_x = generator.normal(size=(4, 3))
check_gamma = generator.normal(size=3)
check_beta = generator.normal(size=3)
check_dout = generator.normal(size=(4, 3))
_, check_cache = batchnorm_forward(
    check_x, check_gamma, check_beta, {"mode": "train"}
)
analytical_dx, analytical_dgamma, analytical_dbeta = batchnorm_backward(
    check_dout, check_cache
)
numerical_dx = eval_numerical_gradient_array(
    lambda candidate: batchnorm_forward(
        candidate, check_gamma, check_beta, {"mode": "train"}
    )[0],
    check_x, check_dout,
)
numerical_dgamma = eval_numerical_gradient_array(
    lambda candidate: batchnorm_forward(
        check_x, candidate, check_beta, {"mode": "train"}
    )[0],
    check_gamma, check_dout,
)
numerical_dbeta = eval_numerical_gradient_array(
    lambda candidate: batchnorm_forward(
        check_x, check_gamma, candidate, {"mode": "train"}
    )[0],
    check_beta, check_dout,
)
errors = {
    "dx": relative_error(analytical_dx, numerical_dx),
    "dgamma": relative_error(analytical_dgamma, numerical_dgamma),
    "dbeta": relative_error(analytical_dbeta, numerical_dbeta),
}
print(errors)
assert all(error < 1e-8 for error in errors.values())

{'dx': 4.181787121944117e-11, 'dgamma': 2.06498855317384e-12, 'dbeta': 4.972988090406063e-12}


## 4. Inverted dropout

Dropout randomly removes activations during training. Let $p$ be the probability of **keeping** a unit. A mask is sampled independently:

$$
M_i\sim\operatorname{Bernoulli}(p).
$$

In inverted dropout, training output is

$$
Y_i=X_i\frac{M_i}{p}.
$$

Because $\mathbb{E}[M_i]=p$, the expected output equals the input. Inference can therefore use `Y = X` with no random masking or additional scaling.

The backward pass uses the same scaled mask sampled during the forward pass. A new mask would describe a different computational graph.

### Exercise 4 — Implement dropout and check its expectation

Complete training/inference forward behavior and backward masking. Then average many stochastic passes to verify that inverted dropout preserves activations in expectation.

In [10]:
def dropout_forward(x, *, keep_probability, mode, generator):
    """Apply inverted dropout and return a cache for backward."""
    if not 0.0 < keep_probability <= 1.0:
        raise ValueError("keep_probability must be in (0, 1]")
    if mode == "train":
        # Keep each activation with probability p, then scale retained values by 1 / p.
        # The supplied generator makes the random behavior reproducible.
        mask = (generator.random(x.shape) < keep_probability) / keep_probability
        output = x * mask
    elif mode == "test":
        mask = None
        # Inverted dropout already handles scaling during training.
        output = x
    else:
        raise ValueError("mode must be 'train' or 'test'")
    return output, (mode, mask)


def dropout_backward(dout, cache):
    """Backpropagate through the same mask used during forward."""
    mode, mask = cache
    # Training reuses the exact scaled mask from the corresponding forward pass.
    return dout * (mask if mode == "train" else 1)


dropout_x = np.linspace(0.5, 2.0, 12).reshape(3, 4)
dropout_generator = np.random.default_rng(SEED)
samples = np.stack([
    dropout_forward(
        dropout_x, keep_probability=0.6, mode="train",
        generator=dropout_generator,
    )[0]
    for _ in range(10_000)
])
mean_output = samples.mean(axis=0)
test_output, test_cache = dropout_forward(
    dropout_x, keep_probability=0.6, mode="test",
    generator=dropout_generator,
)
assert np.allclose(mean_output, dropout_x, rtol=0.03, atol=0.03)
assert np.array_equal(test_output, dropout_x)
assert np.array_equal(dropout_backward(np.ones_like(dropout_x), test_cache), np.ones_like(dropout_x))
print("Maximum expectation error:", np.max(np.abs(mean_output - dropout_x)))

Maximum expectation error: 0.01890909090923354


## Normalization and dropout are not interchangeable

Batch normalization primarily changes activation scaling and the optimization landscape, although minibatch statistics also introduce noise. Dropout explicitly removes random activations to discourage reliance on particular paths. Using one does not automatically provide the behavior of the other.

Both require correct mode handling:

- batch normalization uses batch statistics in training and running statistics in inference;
- dropout samples masks in training and becomes the identity in inference.

## Reflection

1. **What distinguishes batch, layer, instance, and group normalization?**  
They normalize over different axes. Batch normalization uses examples in a minibatch; layer normalization uses the features of each example; instance normalization treats each channel of each image separately; and group normalization divides an image's channels into groups.

2. **Why do batch-normalization training and inference modes differ?**  
Training uses the current minibatch statistics and updates running statistics. Inference uses the stored running statistics so that a prediction does not depend on other examples in a batch.

3. **What roles do $\gamma$ and $\beta$ play?**  
$\gamma$ scales and $\beta$ shifts each normalized feature. They are learned parameters that let the network choose the most useful scale and location after normalization.

4. **Why should numerical checks use a fresh batch-normalization state?**  
Each numerical-gradient evaluation should represent the same function. Reusing mutable state would update the running statistics between evaluations and could make the measured gradient unreliable.

5. **Why does inverted dropout divide by the keep probability during training?**  
Dividing retained activations by $p$ compensates for the fraction that are removed, so the expected activation remains unchanged.

6. **Why must dropout backward reuse the forward mask?**  
The mask is part of that particular forward computational graph. Dropped activations contributed nothing to the output, so they must also receive zero gradient.

7. **Why is dropout disabled during inference?**  
Inference should be deterministic and use the complete network. With inverted dropout, the necessary scaling was already applied during training, so inference is simply the identity operation.

8. **When might batch normalization be inconvenient compared with layer or group normalization?**  
It can be inconvenient with very small or variable batch sizes because batch statistics become noisy or unavailable. Layer and group normalization do not depend on other examples in the minibatch.

After this notebook, the verified functions can be moved into reusable normalization and regularization modules before comparing them inside a CNN.